# Research Question & Provisional Lane

**Intern:** Ayesha  
**Track:** Machine Learning  
**Item:** ML-02 — Research Question and Provisional Lane

---

## Provisional Lane: Ranking Signal Analysis

I'm choosing this lane because it lets me study which safe, observable content and search signals are associated with visibility, clicks, engagement, or ranking movement — without needing to make causal claims about Google's algorithm. It fits an evidence-first, correlation-style approach, which matches what I've already found while running the Week 1-2 starter notebooks.

## Research Question

**Which content and search-behavior signals show the strongest association with a page's visibility and engagement, and can these signals be combined into a reliable, evidence-backed ranking of pages worth reviewing first?**

## The Four Things This Question Connects

- **The data I have:** the anonymized starter dataset (~30,000 pages) with observable signals — impressions, clicks, position, content age, word count, update recency, trend direction — and the full ~79M-row warehouse coming in Week 3.
- **The decision it improves:** which pages a content team should prioritize reviewing or refreshing, instead of guessing or reviewing pages in an arbitrary order.
- **The action someone takes:** a content strategist pulls the ranked list and starts with the top of it — review, refresh, or leave alone.
- **The cost of getting it wrong:** time is spent reviewing pages that were never going to move the needle, while genuinely declining or high-opportunity pages go unreviewed for longer.

## Unit of Analysis

**Per page** (individual URL/row in the dataset). The starter dataset is already structured this way (one row = one page), and it lets recommendations point at something someone can actually act on.

## Output

A ranked list of pages, each with a score and a plain-language reason code (e.g. "stale + still visible", "high impressions, low CTR"), plus a short signal report describing which signals actually correlate with outcomes and which popular assumptions don't hold up in the data.

## Why Data/ML Can Help Here

A few beliefs that feel intuitively true turn out not to hold in the data (see numbers below) — which means a hand-written rule based on gut feeling will systematically misprioritize pages. A model (even a simple, readable one, like the depth-2/3 decision tree from Week 2) can pick up on real, non-obvious combinations of signals instead of relying on assumptions that don't check out.

## Backing This Up: 2-3 Real Numbers From the Starter Dataset

These are the same discoveries from the Week 1-2 notebooks, reproduced here to justify the lane choice.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "pages,", df.shape[1], "columns")

### Number 1 — Search volume barely predicts actual traffic

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> a common assumption ('high search volume = more traffic') does not hold here.")

### Number 2 — CTR collapses sharply by position

This shows position/visibility signals carry real, usable structure — worth building a ranking around.

In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())

### Number 3 — A simple hand-rule (stale x visible) already ranks well at the top

This is evidence that safe, observable signals *do* carry real predictive value for a ranking use case — worth refining into a proper model rather than starting from scratch.

In [ ]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

stale   = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible_flag * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
print(f"Hand rule Precision@20: {precision_at_k(df['hand_rule_score'], y, 20):.3f}")
print(f"Hand rule Precision@50: {precision_at_k(df['hand_rule_score'], y, 50):.3f}")

## Next Steps

1. Review the **Lane Example Cuts** and **FlyRank Warehouse Dataset** (Hugging Face, gated) once Week 3 access opens.
2. Use starter notebook 03 (DuckDB over `hf://` + sklearn) to expand feature exploration to the full warehouse.
3. Refine features and the ranking model beyond the depth-2/3 decision tree — possibly random forest, as already demoed in the reference pipeline (Precision@50 ~0.74 there).
4. Lane lock is due end of Week 4 — this pick may still shift if the fuller warehouse data suggests a stronger fit elsewhere (e.g. Refresh/Content Opportunity Scoring).

## Note on Scope

This is a provisional statement, not a final commitment. All data shown here comes from the small, public-safe anonymized starter dataset — no client names, domains, URLs, or private data are included in this or future public documents. Everything above is framed as *observed* / *directional*, not as proof of how Google's ranking algorithm works.

---
*Built on the FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai)*